# 004 · Indentation and Code Blocks

> **Syntax › Basics** · Python 3.14

The most visible difference between C# and Python: there are no braces. In Python, **indentation is syntax** — the way you indent a block is how the interpreter knows where it starts and ends.

**In this notebook you will learn:**

- how a colon plus indentation replaces `{ ... }`
- the 4-space convention and what happens when indentation is wrong (`IndentationError`)
- how to write an empty block with `pass`
- how to split long lines (implicit continuation inside brackets, and the backslash)
- why you never need semicolons — and why you shouldn't use them

## C# mental model

| C# | Python |
|---|---|
| `{` opens a block | `:` at the end of the header line, then indent |
| `}` closes a block | dedent back to the previous level |
| indentation is cosmetic | indentation is **mandatory** and meaningful |
| `{ }` empty block | `pass` |
| `;` terminates every statement | a newline ends a statement; no `;` needed |
| a statement may span lines freely | lines continue only inside `()`, `[]`, `{}` or after `\` |

## Colon + indentation instead of braces

Every compound statement — `if`, `for`, `while`, `def`, `class`, `try`, `with`, `match` — has a **header line ending in a colon**, followed by an indented **block**. The block ends when the indentation returns to the header's level.

```csharp
// C#
if (temperature > 30)
{
    Console.WriteLine("Hot");
    Console.WriteLine("Stay hydrated");
}
Console.WriteLine("Done");
```

In [ ]:
temperature = 35

if temperature > 30:
    print("Hot")
    print("Stay hydrated")   # same indentation -> same block
print("Done")                # dedented -> outside the if

Notice also: no parentheses are required around the condition. `if (temperature > 30):` works, but it is not idiomatic.

Blocks nest by indenting further:

In [ ]:
def classify(numbers):
    for n in numbers:
        if n % 2 == 0:
            print(n, "is even")
        else:
            print(n, "is odd")
    print("checked", len(numbers), "numbers")

classify([1, 2, 3])

⚠️ **Gotcha:** in C#, indentation can lie — the compiler only looks at braces. In Python, indentation *is* the structure, so moving a line one level changes what the code does:

In [ ]:
items = ["a", "b", "c"]

for item in items:
    print("processing", item)
    print("finished")          # inside the loop: printed 3 times

print("-" * 20)

for item in items:
    print("processing", item)
print("finished")              # outside the loop: printed once

## Four spaces, never tabs

PEP 8 says: **use 4 spaces per indentation level**. Any consistent amount technically works within one block, but 4 spaces is universal — configure your editor to insert spaces when you press Tab (VS Code, PyCharm and Rider do this for Python by default).

Python 3 refuses to guess when tabs and spaces are mixed inconsistently and raises `TabError`.

## `IndentationError`

Wrong indentation is a **syntax** error, detected when the code is compiled — before any of it runs. We can demonstrate it safely by passing source code to the built-in `compile()` inside `try/except`:

In [ ]:
examples = {
    "missing indent": "if True:\nprint('hi')\n",
    "unexpected indent": "x = 1\n    y = 2\n",
    "inconsistent dedent": "if True:\n    x = 1\n  y = 2\n",
}

for label, source in examples.items():
    try:
        compile(source, "<example>", "exec")
    except IndentationError as e:
        print(f"{label:20} -> {type(e).__name__}: {e}")

`IndentationError` is a subclass of `SyntaxError` (and `TabError` is a subclass of `IndentationError`):

In [ ]:
print(issubclass(IndentationError, SyntaxError))
print(issubclass(TabError, IndentationError))

## Empty blocks: `pass`

A block can't be empty — there must be at least one statement. Where C# uses `{ }`, Python uses the `pass` statement, which does nothing.

```csharp
// C#
class PlaceholderService { }

void NotImplementedYet() { }

try { RiskyCall(); }
catch (TimeoutException) { }   // deliberately ignore
```

In [ ]:
class PlaceholderService:
    pass

def not_implemented_yet():
    pass

try:
    int("not a number")
except ValueError:
    pass                       # deliberately ignore

print(not_implemented_yet())   # a function that does nothing returns None

In [ ]:
try:
    compile("def empty():\n", "<example>", "exec")
except SyntaxError as e:
    print(f"{type(e).__name__}: {e}")

You will also see the `...` (Ellipsis) literal used as a placeholder body, especially in type stubs and protocol definitions. A docstring alone also counts as a body.

In [ ]:
def todo():
    ...

def documented():
    """A docstring is a statement, so it counts as a body."""

print(todo(), documented())

## Long lines: continuation

A newline normally ends a statement. To split a long statement across lines, prefer **implicit continuation**: inside any open `(`, `[` or `{`, newlines are ignored until the bracket closes. Indentation inside the brackets is free-form (it's for readability only).

```csharp
// C# — statements end at ';', so you can break lines anywhere
var total = subtotal
          + shipping
          - discount;
```

In [ ]:
subtotal, shipping, discount = 100, 5, 10

total = (subtotal
         + shipping
         - discount)

colors = [
    "red",
    "green",
    "blue",     # a trailing comma is allowed (and recommended)
]

print(total, colors)

The backslash `\` explicitly continues a line. It works, but it is fragile — a single space after the `\` is a syntax error — so PEP 8 prefers brackets. The main legitimate use is long `with` or `assert` statements in older code.

In [ ]:
total = subtotal + \
        shipping - \
        discount
print(total)

⚠️ **Gotcha:** without brackets or a backslash, a line break ends the statement. This is not an error but changes the meaning:

In [ ]:
result = 1 + 2
+ 3              # a separate statement (unary plus on 3) — evaluated and discarded!
print(result)    # 3, not 6

## No semicolons needed

Statements end at the newline, so C#'s `;` is not needed. Python *does* allow `;` to put several simple statements on one line, but PEP 8 discourages it — one statement per line is the norm.

In [ ]:
x = 1; y = 2; print(x + y)    # legal but unidiomatic

x = 1
y = 2
print(x + y)                  # preferred

A trailing `;` after a single statement is allowed too, but it's just noise — linters flag it. (In Jupyter, a trailing `;` on the last line suppresses the display of its value, which is the one place you'll see it used deliberately.)

Similarly, a *simple* one-statement body may sit on the header line, `if ok: print("yes")`, but PEP 8 discourages this as well.

## Summary

| Concept | C# | Python |
|---|---|---|
| Start a block | `{` | `:` + newline + indent |
| End a block | `}` | dedent |
| Indentation | cosmetic | syntax (4 spaces per level) |
| Empty block | `{ }` | `pass` (or `...`) |
| Bad indentation | compiles fine | `IndentationError` / `TabError` |
| Continue a line | anywhere (until `;`) | inside `()`, `[]`, `{}` — or `\` |
| End a statement | `;` | newline |
| Several statements per line | common | `a; b` — legal but discouraged |

## Exercises

1. Translate this C# into Python:
   ```csharp
   foreach (var n in new[] { 3, 8, 12 })
   {
       if (n > 5) { Console.WriteLine($"{n} is big"); }
       else { }
   }
   ```
2. Using `compile()` and `try/except`, show that the source `"for i in range(3):\nprint(i)\n"` is rejected, and print the error type and message.
3. Rewrite this backslash-continued expression to use implicit continuation instead:
   ```python
   is_valid = len(name) > 0 and \
              len(name) < 50 and \
              name.isalpha()
   ```

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
# 1 — the empty else needs `pass`, or (better) just drop it
for n in [3, 8, 12]:
    if n > 5:
        print(f"{n} is big")
    else:
        pass

# 2
try:
    compile("for i in range(3):\nprint(i)\n", "<example>", "exec")
except IndentationError as e:
    print(f"{type(e).__name__}: {e}")

# 3 — wrap the whole expression in parentheses
name = "alice"
is_valid = (len(name) > 0
            and len(name) < 50
            and name.isalpha())
print(is_valid)
```

</details>